# 04 · Cost, caching and callbacks

Tutorial sections 8–10 ([`TUTORIAL.md`](../TUTORIAL.md)).

All models are reached through AWS Bedrock. Run the setup cell first: it loads `.env.local` next to this notebook (AWS profile, region, model choice, `LIVE` switch) and prints those settings. Credentials come from your AWS SSO profile, never from the notebook or the file. With `LIVE` unset or `0`, every call returns a mock response: no AWS access, no cost. See `00_credentials.ipynb` to set up.

`FAST`, `SMART` and `ALT` are the three Bedrock models from `.env.local` (defaults: Claude Haiku 5.5, Claude Opus 5.5, Amazon Nova 2 Lite).

In [1]:
from _env import ALT, FAST, LIVE, MODELS, SMART, mock, status  # loads .env.local; import before litellm

import litellm

litellm.suppress_debug_info = True
status()

env file    : .env.local (found)
mode        : LIVE (real Bedrock calls, real cost)
AWS_PROFILE : set (from .env.local)
AWS_REGION  : us-west-2
MODEL_FAST  : bedrock/us.anthropic.claude-haiku-5-5 (default)
MODEL_SMART : bedrock/us.anthropic.claude-opus-5-5 (default)
MODEL_ALT   : bedrock/us.amazon.nova-2-lite-v1:0 (default)


## 8. Counting tokens, pricing calls and capping spend

[Section 8](../TUTORIAL.md#8-counting-tokens-pricing-calls-and-capping-spend)

In [2]:
MODEL = SMART
messages = [{"role": "user", "content": "Summarise the plot of Hamlet. " * 20}]

print("prompt tokens (estimate):", litellm.token_counter(model=MODEL, messages=messages))
print("context window:", litellm.get_model_info(MODEL)["max_input_tokens"])

resp = litellm.completion(model=MODEL, messages=messages, max_tokens=1024,
                          **mock("A prince seeks revenge. " * 30))
print(f"this call cost ${litellm.completion_cost(resp):.6f}")

for model in MODELS:
    info = litellm.get_model_info(model)
    p, c = info["input_cost_per_token"] * 1e6, info["output_cost_per_token"] * 1e6
    print(f"{model:45} ${p:7.2f} per 1M in  ${c:7.2f} per 1M out")

prompt tokens (estimate): 188
context window: 1000000
this call cost $0.023795
bedrock/us.anthropic.claude-haiku-5-5         $   0.11 per 1M in  $   0.55 per 1M out
bedrock/us.anthropic.claude-opus-5-5          $   4.40 per 1M in  $  22.00 per 1M out
bedrock/us.amazon.nova-2-lite-v1:0            $   0.33 per 1M in  $   2.75 per 1M out


A spend cap you control. With `LIVE=1` this loop really spends up to the limit (about $0.001 plus one call). Prices are LiteLLM's bundled Bedrock list prices; your AWS bill is the source of truth.

In [3]:
class BudgetGuard:
    """Refuse calls once recorded spend reaches the limit."""

    def __init__(self, limit_usd):
        self.limit, self.spent = limit_usd, 0.0

    def completion(self, **kwargs):
        if self.spent >= self.limit:
            raise RuntimeError(f"budget ${self.limit} reached (spent ${self.spent:.6f})")
        resp = litellm.completion(**kwargs)
        self.spent += litellm.completion_cost(resp)
        return resp


guard = BudgetGuard(limit_usd=0.001)
for i in range(1, 100):
    try:
        guard.completion(model=MODEL, messages=messages, max_tokens=1024,
                         **mock("A prince seeks revenge. " * 30))
    except RuntimeError as e:
        print(f"call {i} refused: {e}")
        break

call 2 refused: budget $0.001 reached (spent $0.023795)


## 9. Caching

[Section 9](../TUTORIAL.md#9-caching)

In [4]:
from litellm.caching.caching import Cache

litellm.cache = Cache(type="local")
messages = [{"role": "user", "content": "What is the capital of Australia?"}]

for attempt in (1, 2):
    resp = litellm.completion(model=FAST, messages=messages, max_tokens=1024,
                              caching=True, **mock("Canberra."))
    print(f"call {attempt}: {resp.choices[0].message.content!r}  id={resp.id}  "
          f"cache_hit={resp._hidden_params.get('cache_hit')}")

resp = litellm.completion(model=FAST, messages=messages, max_tokens=1024, caching=True,
                          cache={"no-cache": True}, **mock("Canberra."))
print("call 3 with no-cache: cache_hit =", resp._hidden_params.get("cache_hit"))

litellm.cache = None  # global setting: switch it off so later cells aren't cached

call 1: "The capital of Australia is **Canberra**. It was selected as a compromise between Sydney and Melbourne, the country's two largest cities at the time, and became the seat of government in 1927."  id=chatcmpl-f1fc180f-5199-4a06-b131-9451890f117c  cache_hit=None
call 2: "The capital of Australia is **Canberra**. It was selected as a compromise between Sydney and Melbourne, the country's two largest cities at the time, and became the seat of government in 1927."  id=chatcmpl-f1fc180f-5199-4a06-b131-9451890f117c  cache_hit=True
call 3 with no-cache: cache_hit = None


## 10. Callbacks: logging, metrics and cost attribution

The failing call is always simulated. [Section 10](../TUTORIAL.md#10-callbacks-logging-metrics-and-cost-attribution)

In [5]:
from litellm.integrations.custom_logger import CustomLogger


class UsageLog(CustomLogger):
    def __init__(self):
        self.total = 0.0

    def log_success_event(self, kwargs, response_obj, start_time, end_time):
        cost = kwargs.get("response_cost") or 0.0
        self.total += cost
        tags = kwargs["litellm_params"]["metadata"].get("tags")
        print(f"[ok]   {kwargs['model']:36} {(end_time - start_time).total_seconds() * 1000:5.0f}ms "
              f"{response_obj.usage.total_tokens:4} tok  ${cost:.6f}  tags={tags}")

    def log_failure_event(self, kwargs, response_obj, start_time, end_time):
        exc = kwargs.get("exception")
        print(f"[fail] {kwargs['model']:36} {type(exc).__name__}: {str(exc)[:50]}")


log = UsageLog()
litellm.callbacks = [log]

msgs = [{"role": "user", "content": "Give me a fun fact."}]
litellm.completion(model=FAST, messages=msgs, max_tokens=1024,
                   metadata={"tags": ["demo"]}, **mock("Octopuses have three hearts."))
litellm.completion(model=ALT, messages=msgs, max_tokens=1024,
                   metadata={"tags": ["demo"]}, **mock("Honey never spoils."))
try:
    litellm.completion(model=SMART, messages=msgs, mock_response=litellm.RateLimitError(
        "ThrottlingException", llm_provider="bedrock", model=SMART))
except litellm.RateLimitError:
    pass
print(f"\nrunning total: ${log.total:.6f}")

litellm.callbacks = []

[ok]   us.anthropic.claude-haiku-5-5         1813ms   95 tok  $0.000045  tags=['demo']
[ok]   us.amazon.nova-2-lite-v1:0            2878ms  296 tok  $0.000688  tags=['demo']
[fail] us.anthropic.claude-opus-5-5         RateLimitError: litellm.RateLimitError: ThrottlingException

running total: $0.000733
